# Race Model Workshop — Single Session: Solved Version

This solved copy keeps the full teaching notes from the student version, but the exercise cells contain one reference implementation.

This workshop applies the **race model inequality** (Miller 1982) to one IMRF experiment session.
You will implement the core statistical operations step by step — modality parsing,
latency filtering, empirical CDF, Miller bound, and violation area — then apply
them to either Unity button RT or AOI-derived time to first fixation.

With a single participant this is a *quality check* and *demonstration*, not
a group-level statistical test.


## Learning Goals

1. Filter a reaction-time array to a valid range.
2. Compute an empirical cumulative distribution function (ECDF).
3. Compute the Miller upper bound from two unimodal CDFs.
4. Quantify the positive violation area (race-model breach) with the trapezoid rule.
5. Compare button-response and gaze-first-fixation latency sources.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import pyxdf
import matplotlib.pyplot as plt
from IPython.display import display

from nb_setup import DEFAULT_PARTICIPANT_ID, ensure_workshop_data, setup_repo
REPO_ROOT, EYE_ROOT = setup_repo()
ensure_workshop_data(EYE_ROOT, require_behavioral=True)

from libs.project_config import PLOT_COLORS
from libs.analysis.recording_helpers import find_neon_recording, neon_participant_id
from libs.analysis.race_model_utils import (
    ecdf,
    cdf_on_grid,
    make_common_grid,
    miller_bound,
    compute_all_models,
)

Repo root: /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking
Workshop data ready: /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/data_output


## Pre-solved — Filter Valid Reaction Times

Before analysing RT distributions we remove outliers: anticipatory responses
(too fast, < 100 ms) and missed or late responses (> 2000 ms). We also
discard NaN values.

Complete `filter_valid_rts`. Keep only finite RT values in the closed interval
`[rt_min, rt_max]`.


In [4]:
def filter_valid_rts(
    rt_ms: np.ndarray,
    rt_min: float = 100.0,
    rt_max: float = 2000.0,
) -> np.ndarray:
    """Keep only finite RTs in [rt_min, rt_max]."""
    rt_ms = np.asarray(rt_ms, dtype=float)
    mask  = np.isfinite(rt_ms) & (rt_ms >= rt_min) & (rt_ms <= rt_max)
    return rt_ms[mask]

In [5]:
def test_filter_valid_rts(fn):
    rts = np.array([50.0, 100.0, 500.0, 2000.0, 2500.0, np.nan, np.inf, -100.0])
    out = fn(rts, rt_min=100.0, rt_max=2000.0)
    assert set(out) == {100.0, 500.0, 2000.0}
    assert len(fn(np.array([100.0, 2000.0]), 100.0, 2000.0)) == 2
    assert len(fn(np.array([np.nan, 5.0, 9999.0]), 100.0, 2000.0)) == 0
    print("Pre-solved RT filter passed.")


test_filter_valid_rts(filter_valid_rts)

Pre-solved RT filter passed.


## Pre-solved — Empirical Cumulative Distribution Function

The race model is expressed in terms of CDFs. The **empirical CDF** (ECDF)
estimates F(t) = P(RT ≤ t) from a finite sample: sort the n observed RTs and
assign probability i/n to the i-th sorted value.

$$\hat{F}(x_{(i)}) = \frac{i}{n}, \qquad i = 1, \ldots, n$$

Complete `compute_ecdf`. The function must:
1. Discard non-positive and non-finite values (the race model is defined for
   positive RT only).
2. Sort the remaining values.
3. Assign probabilities 1/n, 2/n, … 1 to the sorted values.


In [6]:
def compute_ecdf(rts: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Compute the empirical CDF from a 1-D RT array."""
    rts = np.asarray(rts, dtype=float).ravel()
    rts = rts[np.isfinite(rts) & (rts > 0)]
    if rts.size == 0:
        return np.array([]), np.array([])
    xs = np.sort(rts)
    Fs = np.arange(1, xs.size + 1) / xs.size
    return xs, Fs

In [7]:
def test_compute_ecdf(fn):
    xs, Fs = fn(np.array([300.0, 100.0, 200.0]))
    assert len(xs) == len(Fs) == 3
    assert np.array_equal(xs, [100.0, 200.0, 300.0])
    assert np.allclose(Fs, [1/3, 2/3, 1.0])
    assert Fs[-1] == 1.0

    xs2, _ = fn(np.array([-50.0, 0.0, np.nan, 400.0, 200.0]))
    assert len(xs2) == 2

    xe, Fe = fn(np.array([]))
    assert len(xe) == 0 and len(Fe) == 0

    print("Pre-solved ECDF helper passed.")


test_compute_ecdf(compute_ecdf)

Pre-solved ECDF helper passed.


## Exercise 10 — Miller Upper Bound

Under the race model, the fastest channel (A or V) determines response time on
every trial. Miller (1982) showed that this places an **upper bound** on how
fast the bimodal CDF can rise:

$$F_{\text{bound}}(t) = \min\bigl(1,\; F_A(t) + F_V(t)\bigr)$$

If the observed F_AV exceeds this bound at any time t, multisensory
integration cannot be explained by a simple race: the two channels must
interact.

Complete `miller_upper_bound`. Both input arrays are already evaluated on the
same time grid.


In [8]:
def miller_upper_bound(
    Fa: np.ndarray,
    Fv: np.ndarray,
) -> np.ndarray:
    """Compute the Miller race model bound: min(1, F_A + F_V)."""
    return np.minimum(1.0, Fa + Fv)

In [9]:
def test_miller_upper_bound(fn):
    Fa = np.array([0.1, 0.4, 0.7, 1.0])
    Fv = np.array([0.2, 0.3, 0.5, 0.9])
    bound = fn(Fa, Fv)
    assert np.allclose(bound, [0.3, 0.7, 1.0, 1.0])
    assert bound.max() <= 1.0
    assert np.all(np.diff(bound) >= 0)
    assert np.allclose(fn(Fa, Fv), miller_bound(Fa, Fv))
    print("Exercise 10 passed.")


test_miller_upper_bound(miller_upper_bound)

Exercise 11 passed.


## Pre-solved — Positive Violation Area

When the observed bimodal CDF rises *above* the Miller bound, the violation
curve is positive:

$$\text{violation}(t) = F_{\text{AV}}(t) - F_{\text{bound}}(t)$$

The **positive violation area** integrates the part that is above zero — a
scalar that summarises how strongly the data breach the race model:

$$A^+ = \int \max\bigl(\text{violation}(t),\; 0\bigr)\, dt$$

Use `np.trapezoid` (trapezoid rule) to approximate the integral. `np.clip(..., 0,
None)` sets negative values to zero before integrating.


In [10]:
def race_violation_area(
    Fav: np.ndarray,
    bound: np.ndarray,
    t_grid: np.ndarray,
) -> float:
    """Positive area where F_AV exceeds the Miller bound."""
    violation = Fav - bound
    positive  = np.clip(violation, 0, None)
    return float(np.trapezoid(positive, t_grid))

In [11]:
def test_race_violation_area(fn):
    t = np.array([0.0, 1.0, 2.0, 3.0])

    assert fn(np.array([0.1, 0.3, 0.5, 0.7]),
              np.array([0.2, 0.4, 0.6, 0.8]), t) == 0.0

    assert np.isclose(fn(np.ones(4), np.zeros(4), t), 3.0)

    area_part = fn(np.array([0.0, 0.2, 0.8, 0.9]),
                   np.array([0.1, 0.5, 0.5, 0.5]), t)
    assert area_part > 0.0

    print("Pre-solved violation-area helper passed.")


test_race_violation_area(race_violation_area)

Pre-solved violation-area helper passed.


---

## From AOI Fixations to Race-Model Latency

The race model needs one latency per trial. This notebook uses two sources:

| Source | What it captures | Data needed |
|---|---|---|
| **Button RT** | Manual response: stimulus → button press | Unity TrialData CSV (`RT` column) |
| **TTFF** | Oculomotor response: stimulus → first fixation on target AOI | XDF (Lab Recorder) + notebook 03 AOI fixation CSV |

### How TTFF is extracted

Unity sends **LSL markers** to Lab Recorder during the experiment:
`3` = Auditory StimOn, `6` = Visual StimOn, `11` = AV StimOn.
These are recorded in the XDF file alongside the Neon gaze stream.

```
 XDF  ──►  AV_Localization marker stream   (LSL boot clock)
           StimOn markers: 3=A, 6=V, 11=AV
                                │
                                │ align using Neon recording start (info.json)
                                ▼
 Notebook 03 AOI fixation CSV   (Neon Unix nanoseconds ÷ 1e9 = Unix seconds)
 • nearest_target label  (Left / Right)
 • inside_nearest_aoi flag

 For each trial (matched by modality order):
   TTFF = (first fixation with nearest_target == trial_target_aoi
            AND fixation_time > StimOn_time)
          − StimOn_time   →  in ms
```

Comparing TTFF and button RT reveals whether multisensory facilitation
appears first in the **oculomotor** or **manual** response channel.

In [12]:
PARTICIPANT_ID = DEFAULT_PARTICIPANT_ID  # options: "p0096", "p0097", "p0099"

LATENCY_SOURCES = ["button_rt", "ttff"]

# Button RT filtering
BUTTON_ONLY_CORRECT = True
RT_MIN_MS = 100.0
RT_MAX_MS = 2000.0

# TTFF filtering
TTFF_ONLY_CORRECT            = False
TTFF_REQUIRE_BUTTON_RESPONSE = False
TTFF_MIN_MS = 0.0
TTFF_MAX_MS = 2000.0

MIN_TRIALS_PER_CONDITION = 3
MODELS     = ["miller_bound", "independent_race"]
MODALITIES = ["A", "V", "AV"]

## Pre-solved — File Discovery and Trial Loading

Locates data files for the selected participant and loads the trial table.
All paths are deterministic — no datetime matching required.
**Run and continue.**

In [ ]:
# ── Locate files ──────────────────────────────────────────────────────────────
NEON_ROOT = EYE_ROOT / "data_output" / "IMRFSpatialAV" / "neon"
selected_recording = find_neon_recording(NEON_ROOT, participant_id=PARTICIPANT_ID)
if selected_recording is None:
    raise FileNotFoundError(f"No Neon recording found for {PARTICIPANT_ID!r}.")
selected_recording_id = selected_recording.name
SELECTED_PARTICIPANT   = neon_participant_id(selected_recording) or PARTICIPANT_ID

# XDF — one file per participant, deterministic path
xdf_path = (
    EYE_ROOT / "data_output" / "IMRFSpatialAV" / "multimodal"
    / f"sub-{SELECTED_PARTICIPANT}_ses-s001_task-imrfspatialav_multimodal.xdf"
)

# Neon recording metadata
info_path = selected_recording / "info.json"

# AOI fixation CSV produced by notebook 03 (deterministic from recording ID)
aoi_root      = EYE_ROOT / "notebooks" / "reports" / "workshop_aoi"
fixation_path = aoi_root / selected_recording_id / f"aoi_classified_fixations_{selected_recording_id}.csv"

# Unity trial data — newest file matching this participant
behavioral_root = REPO_ROOT / "IMRF_SpatialAVDemo_Unity" / "IMRFDemoData"
trial_files = sorted(behavioral_root.glob(
    f"Subject_{SELECTED_PARTICIPANT}_AVLoc_Data_*/IMRFDemo_TrialData_*.csv"
))
trial_path = trial_files[-1] if trial_files else None

for label, path in [("XDF", xdf_path), ("Fixations", fixation_path)]:
    if path is None or not path.is_file():
        print(f"{label} not found — TTFF cannot be computed: {path}")
if trial_path is None or not trial_path.is_file():
    raise FileNotFoundError(f"No TrialData CSV found for {SELECTED_PARTICIPANT!r}.")

print(f"Participant  : {SELECTED_PARTICIPANT}")
print(f"Recording    : {selected_recording_id}")
print(f"XDF          : {xdf_path.name}")
print(f"TrialData    : {trial_path.name if trial_path else 'not found'}")
print(f"Fixations    : {fixation_path.name if fixation_path.is_file() else 'not found'}")

# ── Trial-loading helpers ─────────────────────────────────────────────────────
def parse_modality(trial_code: str) -> str | None:
    '''Return "AV", "A", "V", or None from a TrialCode string.'''
    code = str(trial_code).lower().strip()
    if code.startswith("av"):        return "AV"
    if re.match(r"^a($|_)", code):  return "A"
    if re.match(r"^v($|_)", code):  return "V"
    return None


def _is_present(series: pd.Series) -> pd.Series:
    return ~series.astype(str).str.strip().str.lower().isin(["", "null", "none", "nan"])


def _target_from_position(value) -> str | float:
    match = re.search(r"\(?\s*([-+]?\d+(?:\.\d+)?)", str(value))
    if not match:
        return np.nan
    x = float(match.group(1))
    return "Left" if x < 0 else "Right" if x > 0 else "Center"


def infer_target_aoi(row: pd.Series) -> str | float:
    side = str(row.get("TrialSide", "")).strip().title()
    if side in {"Left", "Right", "Center"}:
        return side
    for col in ["VisualTargetPosition", "AudioTargetPosition"]:
        value = row.get(col, np.nan)
        if _is_present(pd.Series([value])).iloc[0]:
            target = _target_from_position(value)
            if isinstance(target, str):
                return target
    return np.nan


def standardise_trials(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["modality"]   = out.get("TrialCode", pd.Series(dtype=str)).apply(parse_modality)
    out["target_aoi"] = out.apply(infer_target_aoi, axis=1)
    out["rt_ms"]      = pd.to_numeric(out.get("RT"), errors="coerce") * 1000.0
    out["correct"]    = (
        out["Correct"].astype(str).str.lower().eq("true")
        if "Correct" in out.columns else out["rt_ms"].notna()
    )
    out["did_respond"] = (
        out["DidRespond"].astype(str).str.lower().eq("true")
        if "DidRespond" in out.columns else out["rt_ms"].notna()
    )
    out["side"] = out.get("TrialSide", out["target_aoi"]).astype(str)
    return out


# ── Load ───────────────────────────────────────────────────────────────────────
trials_raw = pd.read_csv(trial_path, sep="\t")
trials     = standardise_trials(trials_raw)
print(f"\nRows: {len(trials):,}")
display(trials[["TrialCode", "modality", "target_aoi", "rt_ms", "did_respond", "correct"]].head())

## Pre-solved — Time to First Fixation (TTFF)

Loads StimOn events from the XDF marker stream and finds the first
target-AOI fixation after each one.

The LSL boot clock (used by Unity markers) is aligned to Neon Unix time by
anchoring on the Neon recording start: `info.json` gives the start in Unix
nanoseconds; the first Neon Gaze sample in the XDF marks the same moment in
LSL time. Their difference is the offset.

In [ ]:
def load_xdf_stim_events(xdf_path: Path, neon_info_path: Path) -> pd.DataFrame:
    '''Load StimOn events from the XDF marker stream, converted to Unix seconds.

    Unity sends integer markers via LSL (3=A, 6=V, 11=AV). Their timestamps
    are in LSL boot-clock time. The offset to Unix seconds is estimated from
    the Neon recording start (info.json) and the first Neon Gaze sample in
    the XDF, which represent the same physical moment in different clocks.
    '''
    data, _ = pyxdf.load_xdf(str(xdf_path))

    # Anchor: first Neon Gaze sample timestamp in LSL time
    lsl_start = None
    for s in data:
        name = (s["info"]["name"] or [""])[0]
        if "Neon" in name and "Gaze" in name and len(s.get("time_stamps", [])):
            lsl_start = float(s["time_stamps"][0])
            break
    if lsl_start is None:
        raise ValueError("Neon Gaze stream not found in XDF.")

    with neon_info_path.open() as f:
        neon_start_unix_s = int(json.load(f)["start_time"]) / 1e9
    lsl_to_unix = neon_start_unix_s - lsl_start

    STIM_CODES = {3: "A", 6: "V", 11: "AV"}
    for s in data:
        name = (s["info"]["name"] or [""])[0]
        if name == "AV_Localization":
            values     = np.array(s["time_series"]).ravel().astype(int)
            timestamps = np.array(s["time_stamps"])
            mask       = np.isin(values, list(STIM_CODES))
            return pd.DataFrame({
                "unix_s":       timestamps[mask] + lsl_to_unix,
                "trigger_code": values[mask],
                "modality":     [STIM_CODES[v] for v in values[mask]],
            })
    raise ValueError("AV_Localization stream not found in XDF.")


def add_ttff_from_xdf(
    trials: pd.DataFrame,
    stim_events: pd.DataFrame,
    aoi_fix_df: pd.DataFrame,
    ttff_max_ms: float = TTFF_MAX_MS,
) -> pd.DataFrame:
    '''Add ttff_ms: first target-AOI fixation (from nb 03) after each StimOn.

    StimOn events are matched to trial rows by modality order — the i-th A
    marker goes with the i-th A row in the TrialData CSV, and so on.
    Fixation timestamps (Unix nanoseconds) are divided by 1e9 to give Unix
    seconds, directly comparable to the converted StimOn times.
    '''
    target_col = "nearest_target" if "nearest_target" in aoi_fix_df.columns else "aoi"
    fix = aoi_fix_df[aoi_fix_df[target_col].isin(["Left", "Right"])].copy()
    ts_col = next((c for c in ["timestamp_ns", "start_time"] if c in aoi_fix_df.columns), None)
    if ts_col is None:
        raise KeyError("AOI fixation CSV has no timestamp column (expected timestamp_ns or start_time).")
    fix["fix_unix_s"] = fix[ts_col] / 1e9
    fix = fix.sort_values("fix_unix_s").reset_index(drop=True)

    max_s   = ttff_max_ms / 1000.0
    ttff_ms = []
    mod_idx = {"A": 0, "V": 0, "AV": 0}

    for _, row in trials.iterrows():
        mod    = row.get("modality")
        target = str(row.get("target_aoi", "")).strip().title()
        if mod not in mod_idx or not target or target.lower() == "nan":
            ttff_ms.append(np.nan)
            continue

        events_mod = stim_events[stim_events["modality"] == mod]
        i = mod_idx[mod]
        mod_idx[mod] += 1

        if i >= len(events_mod):
            ttff_ms.append(np.nan)
            continue

        t0   = events_mod.iloc[i]["unix_s"]
        hits = fix[
            (fix["fix_unix_s"] >= t0) &
            (fix["fix_unix_s"] <= t0 + max_s) &
            (fix[target_col]   == target)
        ]
        ttff_ms.append(
            (hits.iloc[0]["fix_unix_s"] - t0) * 1000.0 if not hits.empty else np.nan
        )

    return trials.assign(ttff_ms=ttff_ms)


if "ttff" in LATENCY_SOURCES and xdf_path.is_file() and fixation_path.is_file():
    stim_events = load_xdf_stim_events(xdf_path, info_path)
    aoi_fix_df  = pd.read_csv(fixation_path)
    trials      = add_ttff_from_xdf(trials, stim_events, aoi_fix_df)
    print(f"Trials with TTFF: {int(trials['ttff_ms'].notna().sum())} / {len(trials)}")
else:
    trials["ttff_ms"] = np.nan
    if "ttff" in LATENCY_SOURCES:
        print("TTFF skipped — XDF or fixation CSV not found.")

cols = ["TrialCode", "modality", "target_aoi", "rt_ms", "ttff_ms"]
display(trials[[c for c in cols if c in trials.columns]].head(10))

## Pre-solved — Latency Readiness

Counts valid trials per condition for each latency source.
`ready_by_source` controls which sources the race model section below will run.

In [16]:
def source_settings(source: str) -> dict:
    if source == "button_rt":
        return {
            "column": "rt_ms",
            "label": "Button RT",
            "min_ms": RT_MIN_MS,
            "max_ms": RT_MAX_MS,
            "only_correct": BUTTON_ONLY_CORRECT,
            "require_button": True,
        }
    if source == "ttff":
        return {
            "column": "ttff_ms",
            "label": "Time to first fixation",
            "min_ms": TTFF_MIN_MS,
            "max_ms": TTFF_MAX_MS,
            "only_correct": TTFF_ONLY_CORRECT,
            "require_button": TTFF_REQUIRE_BUTTON_RESPONSE,
        }
    raise ValueError(f"Unknown latency source: {source}")


def latency_analysis_table(trial_df: pd.DataFrame, source: str) -> pd.DataFrame:
    settings = source_settings(source)
    col = settings["column"]
    if col not in trial_df.columns:
        return trial_df.iloc[0:0].copy()

    values = pd.to_numeric(trial_df[col], errors="coerce")
    keep = trial_df["modality"].isin(MODALITIES) & values.between(
        settings["min_ms"], settings["max_ms"], inclusive="both"
    )
    if settings["require_button"]:
        keep &= trial_df["did_respond"]
    if settings["only_correct"]:
        keep &= trial_df["correct"]

    out = trial_df[keep].copy().reset_index(drop=True)
    out["latency_source"] = source
    out["latency_label"] = settings["label"]
    out["latency_ms"] = pd.to_numeric(out[col], errors="coerce")
    return out


def readiness_for_source(trial_df: pd.DataFrame, analysis_df: pd.DataFrame, source: str) -> pd.DataFrame:
    settings = source_settings(source)
    col = settings["column"]
    base = trial_df.copy()
    base["has_latency"] = pd.to_numeric(base.get(col), errors="coerce").notna()
    readiness = (
        base.groupby("modality", dropna=False)
        .agg(
            n_trials=("modality", "size"),
            n_with_latency=("has_latency", "sum"),
            n_with_button_response=("did_respond", "sum"),
            n_correct=("correct", "sum"),
        )
        .reset_index()
    )
    readiness["latency_source"] = source
    readiness["latency_label"] = settings["label"]
    readiness["n_used_for_race"] = (
        readiness["modality"].map(analysis_df["modality"].value_counts()).fillna(0).astype(int)
    )
    return readiness


analysis_by_source = {}
readiness_rows = []
ready_by_source = {}

for source in LATENCY_SOURCES:
    analysis_by_source[source] = latency_analysis_table(trials, source)
    readiness = readiness_for_source(trials, analysis_by_source[source], source)
    readiness_rows.append(readiness)
    ready_by_source[source] = all(
        len(analysis_by_source[source].loc[analysis_by_source[source]["modality"] == m]) >= MIN_TRIALS_PER_CONDITION
        for m in MODALITIES
    )
    print(f"{source_settings(source)['label']} race-model ready: {ready_by_source[source]}")

readiness_all = pd.concat(readiness_rows, ignore_index=True) if readiness_rows else pd.DataFrame()
display(readiness_all[["latency_source", "modality", "n_trials", "n_with_latency", "n_used_for_race", "n_with_button_response", "n_correct"]])


Button RT race-model ready: True
Time to first fixation race-model ready: True


,latency_source,modality,n_trials,n_with_latency,n_used_for_race,n_with_button_response,n_correct
0,button_rt,A,30,29,29,29,19
1,button_rt,AV,30,30,30,30,25
2,button_rt,V,30,29,29,29,27
3,ttff,A,30,11,11,29,19
4,ttff,AV,30,11,11,30,25
5,ttff,V,30,8,8,29,27


## Latency Summaries


In [17]:
for source, analysis in analysis_by_source.items():
    settings = source_settings(source)
    label = settings["label"]
    if analysis.empty:
        print(f"No valid {label} trials after filtering.")
        continue

    summary = (
        analysis.groupby(["modality", "side"], dropna=False)["latency_ms"]
        .agg(n="count", mean_ms="mean", median_ms="median", sd_ms="std")
        .reset_index()
        .sort_values(["modality", "side"])
    )
    print(label)
    display(summary)

    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    data = [analysis.loc[analysis["modality"] == m, "latency_ms"] for m in MODALITIES]
    ax.boxplot(data, tick_labels=MODALITIES, showmeans=True)
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"Single-session {label} distributions")
    ax.grid(axis="y", alpha=0.25)
    plt.show()


Button RT


,modality,side,n,mean_ms,median_ms,sd_ms
0,A,Left,14,412.657229,411.7599,102.012623
1,A,Right,15,458.722213,395.3705,151.985126
2,AV,Left,15,446.324313,389.7935,278.752836
3,AV,Right,15,435.904113,449.6584,159.116399
4,V,Left,15,395.529887,396.7275,101.441068
5,V,Right,14,406.897721,380.2346,105.566916


Time to first fixation


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50606/1925842483.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,modality,side,n,mean_ms,median_ms,sd_ms
0,A,Left,6,1023.979166,1028.753999,627.650465
1,A,Right,5,1051.797299,1530.569499,789.576790
2,AV,Left,5,1152.913099,1808.585499,970.889212
3,AV,Right,6,1316.598332,1517.846499,652.470047
4,V,Left,5,1005.774699,1474.679499,674.163022
5,V,Right,3,1104.425499,1447.286499,881.844013


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50606/1925842483.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Race Model — Pre-solved ECDF, Exercise 10, And Violation Area Applied

This section runs once per latency source that has enough A, V, and AV trials.
`compute_ecdf` builds the ECDF, `miller_upper_bound` (Exercise 10)
computes the bound, and `race_violation_area` quantifies the breach.


In [18]:
def run_race_model_for_source(source: str, analysis: pd.DataFrame) -> pd.DataFrame:
    settings = source_settings(source)
    label = settings["label"]

    if not ready_by_source.get(source, False):
        counts = {m: int((analysis["modality"] == m).sum()) for m in MODALITIES}
        print(
            f"Skipping {label}: need at least {MIN_TRIALS_PER_CONDITION} valid latencies "
            f"per condition; current counts: {counts}"
        )
        return pd.DataFrame()

    latency_by_mod = {
        m: analysis.loc[analysis["modality"] == m, "latency_ms"].to_numpy(float)
        for m in MODALITIES
    }
    lat_A, lat_V, lat_AV = latency_by_mod["A"], latency_by_mod["V"], latency_by_mod["AV"]
    t_grid = make_common_grid(lat_A, lat_V, lat_AV)

    xa, Fa = compute_ecdf(lat_A)
    xv, Fv = compute_ecdf(lat_V)
    xav, Fav = compute_ecdf(lat_AV)

    Fa_t = cdf_on_grid(xa, Fa, t_grid)
    Fv_t = cdf_on_grid(xv, Fv, t_grid)
    Fav_t = cdf_on_grid(xav, Fav, t_grid)

    bound = miller_upper_bound(Fa_t, Fv_t)
    pos_area = race_violation_area(Fav_t, bound, t_grid)
    violation = Fav_t - bound

    # Production model fits via the library. The argument names say "rt", but
    # the functions operate on any positive latency in milliseconds.
    models = compute_all_models(lat_A, lat_V, lat_AV, t_grid=t_grid, models=MODELS)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    ax = axes[0]
    ax.step(xa, Fa, where="post", label=f"A (n={len(lat_A)})", color=PLOT_COLORS["cdf_A"])
    ax.step(xv, Fv, where="post", label=f"V (n={len(lat_V)})", color=PLOT_COLORS["cdf_V"])
    ax.step(xav, Fav, where="post", label=f"AV (n={len(lat_AV)})", color=PLOT_COLORS["cdf_AV"])
    ax.plot(t_grid, bound, "--", label="Miller bound", color=PLOT_COLORS["cdf_bound"])
    if "independent_race" in models:
        ax.plot(t_grid, models["independent_race"].predicted_cdf, ":",
                label="Independent race", color=PLOT_COLORS["model_independent"])
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("CDF")
    ax.set_title(f"{label}: observed CDFs and race predictions")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

    ax = axes[1]
    ax.plot(t_grid, violation, color=PLOT_COLORS["violation"], label="AV - Miller bound")
    ax.fill_between(t_grid, 0, np.clip(violation, 0, None),
                    color=PLOT_COLORS["violation"], alpha=0.25)
    ax.axhline(0, color="black", lw=1)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("CDF difference")
    ax.set_title(f"Positive area = {pos_area:.3f}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    plt.show()

    rows = [{
        "latency_source": source,
        "latency_label": label,
        "model": "Miller positive area",
        "positive_area": pos_area,
    }]
    for key, result in models.items():
        row = {
            "latency_source": source,
            "latency_label": label,
            "model": result.name,
            "rmse": result.rmse,
            "r_squared": result.r_squared,
            "positive_area": pos_area,
        }
        row.update(result.params)
        rows.append(row)

    model_summary = pd.DataFrame(rows)
    display(model_summary)
    return model_summary


model_summaries = []
for source, analysis in analysis_by_source.items():
    model_summary = run_race_model_for_source(source, analysis)
    if not model_summary.empty:
        model_summaries.append(model_summary)

model_summary_all = pd.concat(model_summaries, ignore_index=True) if model_summaries else pd.DataFrame()


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50606/183308465.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,latency_source,latency_label,model,positive_area,rmse,r_squared
0,button_rt,Button RT,Miller positive area,6.720025,NaN,NaN
1,button_rt,Button RT,Miller Bound,6.720025,0.199315,0.573116
2,button_rt,Button RT,Independent Race,6.720025,0.145666,0.771993


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50606/183308465.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,latency_source,latency_label,model,positive_area,rmse,r_squared
0,ttff,Time to first fixation,Miller positive area,10.536153,NaN,NaN
1,ttff,Time to first fixation,Miller Bound,10.536153,0.447460,-5.157597
2,ttff,Time to first fixation,Independent Race,10.536153,0.330189,-2.352960


## Export


In [19]:
OUTPUT_DIR = EYE_ROOT / "notebooks" / "reports" / "workshop_race"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

combined_out = OUTPUT_DIR / f"{trial_path.stem}_button_rt_ttff_trials.csv"
readiness_out = OUTPUT_DIR / f"{trial_path.stem}_readiness_by_latency_source.csv"
trials.to_csv(combined_out, index=False)
readiness_all.to_csv(readiness_out, index=False)
print(f"Saved combined trial table -> {combined_out}")
print(f"Saved readiness -> {readiness_out}")

for source, analysis in analysis_by_source.items():
    if analysis.empty:
        continue
    analysis_out = OUTPUT_DIR / f"{trial_path.stem}_{source}_clean_trials.csv"
    analysis.to_csv(analysis_out, index=False)
    print(f"Saved {source} clean trials -> {analysis_out}")

if not model_summary_all.empty:
    model_out = OUTPUT_DIR / f"{trial_path.stem}_race_models_by_latency_source.csv"
    model_summary_all.to_csv(model_out, index=False)
    print(f"Saved model summary -> {model_out}")


Saved combined trial table -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_race/IMRFDemo_TrialData_Subject_p0097__2026__06_22__11_05_27_button_rt_ttff_trials.csv
Saved readiness -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_race/IMRFDemo_TrialData_Subject_p0097__2026__06_22__11_05_27_readiness_by_latency_source.csv
Saved button_rt clean trials -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_race/IMRFDemo_TrialData_Subject_p0097__2026__06_22__11_05_27_button_rt_clean_trials.csv
Saved ttff clean trials -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_race/IMRFDemo_TrialData_Subject_p0097__2026__06_22__11_05_27_ttff_clean_trials.csv
Saved model summary -> /Users/eduardo/Workspa

## Short Reflection

1. Why is only the *positive* part of the violation curve used for the area?
2. Compare `button_rt` and `ttff`. Which latency source is race-model ready in
   this session, and why?
3. With a single session of 30 AV trials, how reliable is the positive area
   estimate? What would you need to make it publishable?
4. In the readiness table, why might auditory-only trials have low or missing
   button-response counts, while still having TTFF values?
5. The independent race model always stays below the Miller bound. Why?
   (Hint: compare the two formulas.)
